# Stage 3: Streaming Producer
## ISM 6562 — Final Project — MediStream Telehealth


## Setup

In [ ]:
# kafka-python is not in the base pyspark-notebook image
!pip install -q kafka-python==2.0.2

In [ ]:
import json, time
from datetime import datetime, timezone
from pyspark.sql import SparkSession, functions as F
from kafka import KafkaProducer
from kafka.errors import KafkaError

spark = (SparkSession.builder
    .appName('MediStream-Stage3a-Producer')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '512m')
    .config('spark.driver.memory', '512m')
    .getOrCreate())

KAFKA_BOOTSTRAP = 'kafka:9093'   # internal listener, reachable from the compose network
TOPIC = 'session-metrics'

## Load the session-quality dataset

Prefer curated Parquet (already cleaned). Fall back to the raw `.csv.gz` in landing if curated isn't built yet — keeps this notebook runnable in a fresh environment.

In [ ]:
HDFS_CURATED = 'hdfs://namenode:9000/medistream/curated/session_quality'
HDFS_LANDING = 'hdfs://namenode:9000/medistream/landing/session_quality/'
LOCAL_CURATED = '/home/jovyan/data/curated/session_quality'
LOCAL_LANDING = '/home/jovyan/data/session-quality.csv.gz'

def load_sessions():
    for path, reader in [
        (HDFS_CURATED,  lambda p: spark.read.parquet(p)),
        (LOCAL_CURATED, lambda p: spark.read.parquet(p)),
        (HDFS_LANDING,  lambda p: spark.read.csv(p, header=True, inferSchema=True)),
        (LOCAL_LANDING, lambda p: spark.read.csv(p, header=True, inferSchema=True)),
    ]:
        try:
            df = reader(path)
            print(f'Loaded from {path}')
            return df
        except Exception as e:
            print(f'  miss {path}: {type(e).__name__}')
    raise RuntimeError('No session_quality source found in HDFS or local mount')

sessions = load_sessions()
print(f'Rows: {sessions.count():,}, columns: {sessions.columns}')
sessions.show(5, truncate=False)

## Producer rate decision


In [ ]:
# Producer rate - 50 events/sec for a ~100 second total run with 5000 events.
# Lets the consumer's 2-min windows fill up and slide a few times during the demo.
EVENTS_PER_SEC = 50
MAX_EVENTS     = 5000

assert isinstance(EVENTS_PER_SEC, int) and 1 <= EVENTS_PER_SEC <= 500, \n    'set EVENTS_PER_SEC to an int between 1 and 500'
assert isinstance(MAX_EVENTS, int) and 500 <= MAX_EVENTS <= 50000, \n    'set MAX_EVENTS to an int between 500 and 50000'
print(f'Will publish up to {MAX_EVENTS:,} events at ~{EVENTS_PER_SEC}/sec '
      f'(~{MAX_EVENTS / EVENTS_PER_SEC / 60:.1f} min)')

## Build the event payload and publish

Each event includes the session_id (key), appointment_id (for downstream join), the raw quality metrics, device/OS, and a synthetic `event_time` set to wallclock at publish — that's what the consumer's watermark/window will key off.

In [ ]:
EVENT_COLS = ['session_id', 'appointment_id', 'video_resolution',
              'audio_quality_score', 'latency_ms', 'packet_loss_pct',
              'duration_sec', 'device_type', 'os']

# Pull rows to the driver. With ~250k rows and ~9 cols this is fine; a
# real producer would stream them rather than collect.
rows = (sessions
    .select(*EVENT_COLS)
    .limit(MAX_EVENTS)
    .toPandas()
    .to_dict(orient='records'))
print(f'Materialized {len(rows):,} rows for replay')

producer = KafkaProducer(
    bootstrap_servers=[KAFKA_BOOTSTRAP],
    key_serializer=lambda k: str(k).encode('utf-8'),
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    linger_ms=20,
    acks=1,
)
print('Producer connected')

In [ ]:
interval = 1.0 / EVENTS_PER_SEC
sent = 0
errors = 0
start = time.time()

for row in rows:
    payload = {k: row.get(k) for k in EVENT_COLS}
    payload['event_time'] = datetime.now(timezone.utc).isoformat()
    try:
        producer.send(TOPIC, key=payload['session_id'], value=payload)
        sent += 1
    except KafkaError as e:
        errors += 1
        if errors <= 3:
            print(f'send error: {e}')

    if sent % 500 == 0:
        elapsed = time.time() - start
        rate = sent / max(elapsed, 1e-6)
        print(f'  sent={sent:,}  errors={errors}  elapsed={elapsed:6.1f}s  rate={rate:6.1f}/s')

    time.sleep(interval)

producer.flush()
elapsed = time.time() - start
print(f'\nDone. sent={sent:,} errors={errors} elapsed={elapsed:.1f}s '
      f'avg_rate={sent / max(elapsed, 1e-6):.1f}/s')
producer.close()